# 01. Data Overview

## Prerequisites

This notebook gives a first look at the data used in `transferformance`, including its stucture, coverage, and quality.

### Data Origin

The main data source is [transfermarkt-datasets](https://github.com/dcaribou/transfermarkt-datasets), created and maintained by [David Cereijo](https://github.com/dcaribou), It collects, processes, and catalogues data from the [Transfermarkt website](https://www.transfermarkt.com/), and is distributed in many forms. This project will be using the `.duckdb` database stored in an [author-maintained Cloudflare R2 Public bucket](https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.duckdb).

`transfermarkt-datasets` is made available under the Creative Commons CC0 1.0 Universal Public Domain Dedication.

The file was downloaded with `transferformance.download`, which also records the snapshot's metadata shown below.

In [1]:
import json
from transferformance.configs import PATH_MANIFEST, PATH_RAW_DATA

manifest = json.loads(PATH_MANIFEST.read_text())
display(manifest[PATH_RAW_DATA.name])

{'url': 'https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.duckdb',
 'sha256': 'ba1eff7337b8ca78cb533df0b6eba0d6fc58218e0767460f6770e0b37f5a2113',
 'size_bytes': 210776064,
 'downloaded_at_utc': '2026-11-06T12:02:54.070678+00:00',
 'last_modified': '2026-09-05T09:12:34+00:00'}

### Data Description

*According to `dcaribou/transfermarkt-datasets`'s `README`:*

> The dataset is composed of 12 tables covering competitions, games, clubs, players, appearances, player valuations, club games, game events, game lineups, transfers, countries and national teams. Each table contains the attributes of the entity and IDs that can be used to join them together.

`transfermarkt-datasets`'s ER diagram:

```mermaid
---
config:
  theme: dark
---
erDiagram
    direction LR

    countries {
        int country_id PK
        string country_name
        string confederation
    }
    national_teams {
        int national_team_id PK
        int country_id FK
        string confederation
        int fifa_ranking
    }
    players {
        int player_id PK
        int current_club_id FK
        int current_national_team_id FK
        int international_caps
        int international_goals
    }
    player_valuations {
        int player_id FK
    }
    appearances {
        int appearance_id PK
        int player_id FK
        int game_id FK
    }
    game_events {
        int game_id FK
        int player_id FK
        int club_id FK
    }
    games {
        int game_id PK
        int competition_id FK
        int home_club_id FK
        int away_club_id FK
    }
    club_games {
        int club_id FK
        int opponent_club_id FK
        int game_id FK
    }
    clubs {
        int club_id PK
        int domestic_competition_id FK
    }
    competitions {
        int competition_id PK
        string type
    }

    countries ||--o{ national_teams : has
    national_teams ||--o{ players : includes
    players ||--o{ player_valuations : has
    players ||--o{ appearances : makes
    players ||--o{ game_events : triggers
    appearances }o--|| games : in
    game_events }o--|| games : during
    games ||--o{ club_games : splits_into
    clubs ||--o{ club_games : plays
    clubs ||--o{ players : employs
    clubs ||--o{ game_events : records
    clubs ||--o{ games : plays_in
    competitions ||--o{ games : holds
    competitions ||--o{ clubs : domestic_for
```

### Caveats

According to the `transfermarkt-datasets`'s Github Repo, updates are paused for the time being. The data is current to July 6th, 2026; which includes the FIFA World Cup 2026's Group Stage.

### Decisions

The dataset has 12 tables, but a valuation model doesn't need all of them. To keep the project focused and the feature set small, we selected tables that helps explain a player's market value at any given point in time.

- **Yes:** core tables for the target and features.
- **Optional:** potentially useful, but only added if leakage can be ruled out.
- **No:** out of scope for this version.

| Table | Description | Used | Reason | 
| --- | --- | --- | --- |
| `players` | Player profiles, positions, market values, international caps | Yes | Core part of the project, contains players' static attributes |
| `player_valuations` | Historical market value records | Yes | Core part of the project, target variable |
| `appearances` | One row per player per game played | Yes | Players' stats by season |
| `clubs` | Club details, squad size, market value | Yes | Add clubs and leagues context to the player's valuation |
| `competitions` | Leagues, tournaments and national team competitions | Optional | Use to decide the level of leagues and clubs |
| `transfers` | Player transfers between clubs | Optional | Provide valuable insight about players' actual valuations, but has potential leakage problems | 
| `games` | Match results, lineups, attendance | No | Out of scope |
| `club_games` | Per-club view of each game | No | Out of scope |
| `game_events` | Goals, cards, substitutions | No | Out of scope |
| `game_lineups` | Starting and bench lineups | No | Out of scope |
| `countries` | Country details and confederation membership | No | Not needed |
| `national_teams` | National team profiles, squad size, FIFA ranking | No | Not needed |

## Preperation

### Import libraires

In [2]:
from transferformance.configs import PATH_RAW_DATA

import duckdb
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd

### Import the dataset

In [3]:
con = duckdb.connect(PATH_RAW_DATA, read_only=True)

## Data overview

### `player_valuations` table

#### Basic observations

First, we check out the `player_valuations` table structure and its basic statistics.

In [4]:
print("`player_valuations`'s table structure:")

con.sql("""
    DESCRIBE player_valuations
""").show()

print("`player_valuations`'s table basic statistics:")

con.sql("""
    SUMMARIZE player_valuations
""").show()

`player_valuations`'s table structure:
┌─────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│             column_name             │ column_type │  null   │   key   │ default │  extra  │
│               varchar               │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ player_id                           │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ date                                │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ market_value_in_eur                 │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ current_club_name                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ current_club_id                     │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_club_domestic_competition_id │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────

In [5]:
print("Top 10 player valuations, sorted by `market_value_in_eur` descending.")

con.sql("""
    SELECT *
    FROM player_valuations v
        JOIN LATERAL (
            SELECT p.name
            FROM players p
            WHERE v.player_id = p.player_id
        ) ON TRUE
    ORDER BY market_value_in_eur DESC
    LIMIT 10 
""").show()

Top 10 player valuations, sorted by `market_value_in_eur` descending.
┌───────────┬────────────┬─────────────────────┬─────────────────────┬─────────────────┬─────────────────────────────────────┬─────────────────┐
│ player_id │    date    │ market_value_in_eur │  current_club_name  │ current_club_id │ player_club_domestic_competition_id │      name       │
│   int32   │    date    │        int32        │       varchar       │      int32      │               varchar               │     varchar     │
├───────────┼────────────┼─────────────────────┼─────────────────────┼─────────────────┼─────────────────────────────────────┼─────────────────┤
│    937958 │ 2025-12-12 │           200000000 │ FC Barcelona        │             131 │ ES1                                 │ Lamine Yamal    │
│    342229 │ 2019-06-03 │           200000000 │ Paris Saint-Germain │             583 │ FR1                                 │ Kylian Mbappé   │
│    342229 │ 2019-12-19 │           200000000 │ Paris Saint

**Initial observations:**
- The `player_valuations` table consists of 6 columns: `player_id`, `date`, `market_value_in_eur`, `current_club_name`, `current_club_id`, and `player_club_domestic_competition_id`; having 656,301 entries for 42,892 distinct players. Player valuations span 2000-01-20 to 2026-06-12, which matches the snapshot date noted earlier.
- All columns in the table are completed excepts for `player_club_domestic_competition_id`, which has a 14.17% null value rate.
- Despite having only 5,378 `current_club_id`, there are 8,119 `current_club_name`.
- `market_value_in_eur`seems to be heavily right-skewed: the mean (average) of approximately 2.2 million euros is much higher than the median (q50) of only about 500,000 euro.
- The minimum `market_value_in_eur` is 0 euros.
- Only 33 domestic leagues were covered in the table, showing that the coverage is limited and focused on bigger leagues.

**Follow-ups:**
- The number of current club ids is lesser than the number of club names. Possible causes: the club id and the club name is not sync, or the same club id is used for different teams (B-teams, youth squads, club renames, etc.)
- `player_club_domestic_competition_id` is missing in 14.17% of rows even though `current_club_id` and `current_club_name` are completed. Possible causes: clubs outside the covered leagues, placeholder club values (e.g. "Without Club", "Retired"), or a mapping that was never updated. Check which clubs the nulls belong to.
- `market_value_in_eur` has a minimum of 0. Possible causes: placeholder values, retired players, or players without a club. Check how many zeros exist and where they occur.
- Check for duplicate (`player_id`, `date`) keys and for valuations whose `player_id` is missing from `players`.

#### `current_club_name`, `current_club_id`, `player_club_domestic_competition_id` discrepancies

First, we check the amount of different club names per club id.

In [6]:
con.sql("""
    SELECT
        v.current_club_id,
        c.name AS official_name,
        COUNT(*) AS n_rows,
        COUNT(DISTINCT v.current_club_name) AS n_names,
        list(DISTINCT v.current_club_name)[1:10] AS sample_names
    FROM player_valuations AS v
    LEFT JOIN clubs AS c
        ON c.club_id = v.current_club_id
    GROUP BY v.current_club_id, c.name
    ORDER BY n_names DESC
    LIMIT 10
""").show()

┌─────────────────┬──────────────────────────────┬────────┬─────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ current_club_id │        official_name         │ n_rows │ n_names │                                                                                                sample_names                                                                                                │
│      int32      │           varchar            │ int64  │  int64  │                                                                                                 varchar[]                                                                                                  │
├─────────────────┼──────────────────────────────┼────────┼─────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────

**Observations:** Contrary to our hypothesis that the gap comes from club renames, a single `current_club_id` is attached to many unrelated clubs. For example, the ID that `clubs` maps to `APO Levadiakos Football Club` appears in `player_valuations` with 400 different `current_club_name` values.

**Possible reasons:** the ID and the name may not describe the same point in time (e.g. the name reflects the club at the valuation date while the ID is filled from a different source or snapshot), or the ID may have been assigned incorrectly during data processing. We test this next by inspecting individual valuation histories.

After inspecting several players with `current_club_id = 2672`, we select player with ID `94318` (Jovan Krneta) as a clear example of the pattern. In the player's full valuation history, the club ID stays at 2672 while `current_club_name` changes between unrelated clubs.

In [7]:
con.sql("""
    SELECT 
        p.name AS "player_name", 
        v.date, 
        v.current_club_id, 
        v.current_club_name, 
        c.name AS "actual_club_name",
        v.player_club_domestic_competition_id
    FROM player_valuations v
        JOIN LATERAL (
            SELECT p.name
            FROM players p
            WHERE p.player_id = v.player_id
        ) p ON TRUE
        JOIN LATERAL (
            SELECT c.name
            FROM clubs c
            WHERE c.club_id = v.current_club_id
        ) c ON TRUE
    WHERE v.player_id = 94318
    ORDER BY v.date
""").show(max_rows=40)

┌──────────────┬────────────┬─────────────────┬─────────────────────────────┬──────────────────────────────┬─────────────────────────────────────┐
│ player_name  │    date    │ current_club_id │      current_club_name      │       actual_club_name       │ player_club_domestic_competition_id │
│   varchar    │    date    │      int32      │           varchar           │           varchar            │               varchar               │
├──────────────┼────────────┼─────────────────┼─────────────────────────────┼──────────────────────────────┼─────────────────────────────────────┤
│ Jovan Krneta │ 2010-03-28 │            2672 │ Unknown                     │ APO Levadiakos Football Club │ GR1                                 │
│ Jovan Krneta │ 2010-04-19 │            2672 │ Unknown                     │ APO Levadiakos Football Club │ GR1                                 │
│ Jovan Krneta │ 2012-04-09 │            2672 │ Red Star Belgrade           │ APO Levadiakos Football Club │ GR1      

We now need to check what percetange of player valuations does this phenomenon applies to.

In [8]:
# Is the club ID constant within each player's history?
con.sql("""
    WITH career_track AS (
        SELECT player_id,
               COUNT(DISTINCT current_club_id) AS n_ids,
               COUNT(DISTINCT current_club_name) AS n_names
        FROM player_valuations
        GROUP BY player_id
    )
    SELECT
        COUNT(*) AS n_players,
        SUM((n_ids = 1)::INT) AS one_id,
        SUM((n_ids > 1)::INT) AS multiple_ids,
        SUM((n_ids = 1 AND n_names > 1)::INT) AS one_id_wrong_names,
        SUM((n_ids > 1 AND n_names > 1 AND n_ids <> n_names)::INT) AS mutiple_id_wrong_names,
        SUM((n_ids = n_names)::INT) AS equal_id_counts
    FROM career_track
""").show()

┌───────────┬────────┬──────────────┬────────────────────┬────────────────────────┬─────────────────┐
│ n_players │ one_id │ multiple_ids │ one_id_wrong_names │ mutiple_id_wrong_names │ equal_id_counts │
│   int64   │ int128 │    int128    │       int128       │         int128         │     int128      │
├───────────┼────────┼──────────────┼────────────────────┼────────────────────────┼─────────────────┤
│     41528 │  22707 │        18821 │              17674 │                   1535 │           22313 │
└───────────┴────────┴──────────────┴────────────────────┴────────────────────────┴─────────────────┘



**Observations:** We can see that out of 41,528 players with valuations:
- 17,674 players (42.6%) have their `current_club_id` despite their `current_club_name` changing throughout their career.
- 1,523 players (3.7%) have their `current_club_id` count not maching 1:1 with their `current_club_name`.
- 22,313 players (53.7%) have equal `current_club_id` and `current_club_name` count.

Similarly, looking at the Jovan Krneta career table, we can see that their `player_club_domestic_competition_id` was unchanged (`GR1`, Greece Super League) despite their club progression (`Red Star Belgrade` is a club that plays in Serbia's top domestic league, not Greece's).

In [18]:
con.sql("""
    SELECT competition_id, name, country_name, type
    FROM competitions 
    WHERE competition_id = 'GR1'
""").show()

con.sql("""
    SELECT c.name, c.domestic_competition_id
    FROM clubs c
    WHERE c.name = 'Red Star Belgrade'
""").show()

┌────────────────┬────────────────┬──────────────┬─────────────────┐
│ competition_id │      name      │ country_name │      type       │
│    varchar     │    varchar     │   varchar    │     varchar     │
├────────────────┼────────────────┼──────────────┼─────────────────┤
│ GR1            │ super-league-1 │ Greece       │ domestic_league │
└────────────────┴────────────────┴──────────────┴─────────────────┘

┌───────────────────┬─────────────────────────┐
│       name        │ domestic_competition_id │
│      varchar      │         varchar         │
├───────────────────┼─────────────────────────┤
│ Red Star Belgrade │ SER1                    │
└───────────────────┴─────────────────────────┘



From these examples and findings, we can conclude that the three columns `current_club_id`, `current_club_name` and `player_club_domestic_competition_id` are unreliable, with nearly half of the players with valuations having mismatched club and league information.

**Decision:** 
- `current_club_id`, `current_club_name`, and `player_club_domestic_competition_id` are excluded from the feature set. Club and league context, if added later, will be built from another source (e.g. `appearances`).
- The remaining columns (`player_id`, `date`, `market_value_in_eur`) are kept.